# Hybrid Retrieval and Learning-to-Rank
This notebook demonstrate how to do simple hybrid retrieval using RRF score as well as training LambdaMART model to illustrate how to leverage predictive power using similar features.

## Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

In [ ]:
DATA_DIR = Path("../data/processed")

TRAINING_PATH = DATA_DIR / "train.parquet"
VALIDATION_PATH = DATA_DIR / "validation.parquet"
TEST_PATH = DATA_DIR / "test.parquet"
PRODUCT_CATALOG_PATH = DATA_DIR / "product_catalog.parquet"

RESULTS_DIR = Path("../results")
PRODUCT_EMBEDDINGS_PATH = RESULTS_DIR / "product_embeddings_bge_small.npy"
PRODUCT_UIDS_PATH = RESULTS_DIR / "product_uids.npy"

In [ ]:
validation = pd.read_parquet(VALIDATION_PATH)
product_catalog = pd.read_parquet(PRODUCT_CATALOG_PATH)
train_split = pd.read_parquet(TRAINING_PATH)
test_split = pd.read_parquet(TEST_PATH)

product_embeddings = np.load(PRODUCT_EMBEDDINGS_PATH)
product_uids = np.load(PRODUCT_UIDS_PATH)

## Recreating BM25

In [4]:
import re
from rank_bm25 import BM25Okapi

def tokenize(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return text.split()

tokenized_catalog = product_catalog["product_text"].apply(tokenize).tolist()

bm25 = BM25Okapi(tokenized_catalog)

In [5]:
product_index = pd.Series(
    np.arange(len(product_catalog)),
    index=product_catalog["product_uid"]
)

In [6]:
def get_bm25_scores(query, product_uids):
    query_tokens = tokenize(query)

    indices = product_index.loc[product_uids].to_numpy()

    scores = bm25.get_scores(query_tokens)

    return scores[indices]

## Recreating Dense

In [7]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "BAAI/bge-small-en-v1.5"

model = SentenceTransformer(MODEL_NAME)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

## Prepare the Split 
### Training set

In [8]:
train_queries = train_split["search_term"].unique()

train_query_embeddings = model.encode(
    train_queries.tolist(),
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True,
)
train_query_embeddings_dict = dict(zip(train_queries, train_query_embeddings))

Batches:   0%|          | 0/260 [00:00<?, ?it/s]

### Test set

In [9]:
test_queries = test_split["search_term"].unique()

test_query_embeddings = model.encode(
    test_queries.tolist(),
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True,
)
test_query_embeddings_dict = dict(zip(test_queries, test_query_embeddings))

Batches:   0%|          | 0/33 [00:00<?, ?it/s]

### Validation Set

In [10]:
val_queries = validation["search_term"].unique()

val_query_embeddings = model.encode(
    val_queries.tolist(),
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True,
)
val_query_embeddings_dict = dict(zip(val_queries, val_query_embeddings))

Batches:   0%|          | 0/33 [00:00<?, ?it/s]

## Calculating BM25 and Dense Scores

In [11]:
from tqdm.auto import tqdm

def build_scores(
    ranking_split,
    queries_dict,
    product_embeddings,
):
    all_rows = []

    for query, query_embedding in tqdm(
        queries_dict.items(),
        total=len(queries_dict)
    ):
        query_rows = ranking_split[
            ranking_split["search_term"] == query
        ].copy()

        query_product_uids = query_rows["product_uid"].to_numpy()

        # BM25 scores
        bm25_scores = get_bm25_scores(
            query,
            query_product_uids,
        )
        bm25_scores = np.sort(bm25_scores)[::-1]

        bm25_score_dict = dict(
            zip(query_product_uids, enumerate(bm25_scores, start=1))
        )

        # Dense scores
        dense_uids = query_product_uids

        dense_scores = (
            query_embedding
            @ product_embeddings[
                product_index.loc[dense_uids].to_numpy()
            ].T
        )
        dense_scores = np.sort(dense_scores)[::-1]

        dense_score_dict = dict(
            zip(dense_uids, enumerate(dense_scores, start=1))
        )

        # Build rows
        for _, row in query_rows.iterrows():
            uid = row["product_uid"]

            all_rows.append({
                "search_term": query,
                "product_uid": uid,
                "relevance": row["relevance"],
                "bm25_score": bm25_score_dict.get(uid)[1],
                "bm25_rank": bm25_score_dict.get(uid)[0],
                "dense_score": dense_score_dict.get(uid)[1],
                "dense_rank": dense_score_dict.get(uid)[0]
            })

    return pd.DataFrame(all_rows)

### Training set

In [20]:
train_ltr = build_scores(train_split, train_query_embeddings_dict, product_embeddings)
train_ltr.head()

  0%|          | 0/8316 [00:00<?, ?it/s]

,search_term,product_uid,relevance,bm25_score,bm25_rank,dense_score,dense_rank
0,angle bracket,100001,3.00,13.508624,1,0.706096,1
1,angle bracket,100664,2.67,9.620173,2,0.703637,2
2,angle bracket,100672,2.33,6.258288,3,0.692124,3
3,angle bracket,100739,3.00,4.462849,4,0.691658,4
4,angle bracket,101036,1.33,4.123317,5,0.690503,5


### Test set

In [12]:
test_ltr = build_scores(test_split, test_query_embeddings_dict, product_embeddings)
test_ltr.head()

  0%|          | 0/1040 [00:00<?, ?it/s]

,search_term,product_uid,relevance,bm25_score,bm25_rank,dense_score,dense_rank
0,mdf 3/4,100009,3.00,10.125483,1,0.727892,1
1,mdf 3/4,100865,2.33,9.418430,2,0.719612,2
2,mdf 3/4,107285,2.33,9.338445,3,0.716033,3
3,mdf 3/4,111072,3.00,9.305973,4,0.704373,4
4,mdf 3/4,111098,3.00,9.026350,5,0.678590,5


## Validation set

In [29]:
val_ltr = build_scores(validation, val_query_embeddings_dict, product_embeddings)
val_ltr.head()

  0%|          | 0/1040 [00:00<?, ?it/s]

,search_term,product_uid,relevance,bm25_score,bm25_rank,dense_score,dense_rank
0,deck over,100002,3.00,6.761276,1,0.688096,1
1,deck over,100092,1.67,6.751245,2,0.686087,2
2,deck over,101629,3.00,6.276802,3,0.684301,3
3,deck over,101795,2.67,6.276802,4,0.678518,4
4,deck over,103015,1.67,3.998665,5,0.640633,5


## RRF

In [13]:
# train_ltr["rrf_score"] = 1/(60 + train_ltr["bm25_rank"]) + 1/(60 + train_ltr["dense_rank"])
test_ltr["rrf_score"] = 1/(60 + test_ltr["bm25_rank"]) + 1/(60 + test_ltr["dense_rank"])

## NDCG Formula

In [8]:
from sklearn.metrics import ndcg_score

def calculate_ndcg(query_data, score, k):
    y_true = query_data["relevance"].to_numpy()
    y_score = query_data[score].to_numpy()

    return ndcg_score(
        [y_true],
        [y_score],
        k=k
    )

Calculate NDCG for RRF

In [15]:
rrf_results = []

for query, query_data in test_ltr.groupby("search_term"):
    query_data = query_data.copy()

    rrf_results.append({
        "query": query,
        "num_products": len(query_data),
        "ndcg@3": calculate_ndcg(query_data, "rrf_score", 3),
        "ndcg@5": calculate_ndcg(query_data, "rrf_score", 5),
        "ndcg@8": calculate_ndcg(query_data, "rrf_score", 8)
    })

rrf_results = pd.DataFrame(rrf_results)

In [16]:
print(
    "Mean NDCG@3:",
    rrf_results["ndcg@3"].mean()
)

print(
    "Mean NDCG@5:",
    rrf_results["ndcg@5"].mean()
)

print(
    "Mean NDCG@8:",
    rrf_results["ndcg@8"].mean()
)

Mean NDCG@3: 0.8812594665502764
Mean NDCG@5: 0.9105414019218127
Mean NDCG@8: 0.9398514171813547


Prepare the feature for LTR

In [12]:
relevance_values = sorted(
    train_ltr["relevance"].unique()
)

relevance_to_label = {
    value: i
    for i, value in enumerate(relevance_values)
}

train_ltr["label"] = (
    train_ltr["relevance"]
    .map(relevance_to_label)
    .astype(int)
)

In [22]:
train_ltr.head()

,search_term,product_uid,relevance,bm25_score,bm25_rank,dense_score,dense_rank,label
0,angle bracket,100001,3.00,13.508624,1,0.706096,1,12
1,angle bracket,100664,2.67,9.620173,2,0.703637,2,10
2,angle bracket,100672,2.33,6.258288,3,0.692124,3,8
3,angle bracket,100739,3.00,4.462849,4,0.691658,4,12
4,angle bracket,101036,1.33,4.123317,5,0.690503,5,2


In [30]:
import pickle

with open("ltr_features.pkl", "wb") as f:
    pickle.dump({
        "train_ltr": train_ltr,
        "val_ltr": val_ltr,
        "test_ltr": test_ltr,
    }, f)

In [9]:
import pickle

with open("ltr_features.pkl", "rb") as f:
    ltr_data = pickle.load(f)

train_ltr = ltr_data["train_ltr"]
val_ltr = ltr_data["val_ltr"]
test_ltr = ltr_data["test_ltr"]

In [10]:
features = ["bm25_score", "dense_score"]

In [13]:
X_train = train_ltr[features]
y_train = train_ltr["label"]

X_test = test_ltr[features]
y_test = test_ltr["relevance"].map(relevance_to_label)

X_val = val_ltr[features]
y_val = val_ltr["relevance"].map(relevance_to_label)

In [14]:
train_ltr = train_ltr.sort_values(
    "search_term",
    kind="stable"
).reset_index(drop=True)

test_ltr = test_ltr.sort_values(
    "search_term",
    kind="stable"
).reset_index(drop=True)

val_ltr = val_ltr.sort_values(
    "search_term",
    kind="stable"
).reset_index(drop=True)

In [15]:
group_train = (
    train_ltr.groupby("search_term", sort=False)
    .size()
    .to_numpy()
)

group_test = (
    test_ltr.groupby("search_term", sort=False)
    .size()
    .to_numpy()
)

group_val = (
    val_ltr.groupby("search_term", sort=False)
    .size()
    .to_numpy()
)

In [16]:
import lightgbm as lgb

In [17]:
ranker = lgb.LGBMRanker(
    objective="lambdarank",
    metric="ndcg",
    ndcg_at=[3, 5],
    n_estimators=200,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
)

ranker.fit(
    X_train,
    y_train,
    group=group_train,
    eval_set=[(X_val, y_val)],
    eval_group=[group_val],
)

c:\Users\Kevin\Documents\GitHub\product-search-ranking\.venv\Lib\site-packages\lightgbm\sklearn.py:969: UserWarning: Found 'ndcg_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")
c:\Users\Kevin\Documents\GitHub\product-search-ranking\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000978 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 510
[LightGBM] [Info] Number of data points in the train set: 57854, number of used features: 2


,learning_rate,0.05
,n_estimators,200
,objective,'lambdarank'
,random_state,42
,metric,'ndcg'
,ndcg_at,"[3, 5]"
,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,subsample_for_bin,200000
,class_weight,None


In [18]:
val_ltr = val_ltr.copy()

val_ltr["ltr_score"] = ranker.predict(
    val_ltr[features]
)

c:\Users\Kevin\Documents\GitHub\product-search-ranking\.venv\Lib\site-packages\lightgbm\sklearn.py:969: UserWarning: Found 'ndcg_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


In [19]:
val_ltr["ltr_rank"] = (
    val_ltr
    .groupby("search_term")["ltr_score"]
    .rank(
        method="first",
        ascending=False
    )
)

In [20]:
from sklearn.metrics import ndcg_score
import numpy as np

def evaluate_ndcg(df, score_col):
    ndcg_3 = []
    ndcg_5 = []

    for query, group in df.groupby("search_term"):
        y_true = group["relevance"].to_numpy()
        y_score = group[score_col].to_numpy()

        ndcg_3.append(
            ndcg_score([y_true], [y_score], k=3)
        )

        ndcg_5.append(
            ndcg_score([y_true], [y_score], k=5)
        )

    return np.mean(ndcg_3), np.mean(ndcg_5)

In [21]:
val_ndcg3, val_ndcg5 = evaluate_ndcg(val_ltr, "ltr_score")

print(f"LambdaMART NDCG@3: {val_ndcg3:.6f}")
print(f"LambdaMART NDCG@5: {val_ndcg5:.6f}")

LambdaMART NDCG@3: 0.882820
LambdaMART NDCG@5: 0.909631


Check performance

In [22]:
test_ltr["ltr_score"] = ranker.predict(
    test_ltr[features]
)

test_ltr["ltr_rank"] = (
    test_ltr
    .groupby("search_term")["ltr_score"]
    .rank(
        method="first",
        ascending=False
    )
)

c:\Users\Kevin\Documents\GitHub\product-search-ranking\.venv\Lib\site-packages\lightgbm\sklearn.py:969: UserWarning: Found 'ndcg_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


In [23]:
train_ltr["ltr_score"] = ranker.predict(
    train_ltr[features]
)

train_ltr["ltr_rank"] = (
    train_ltr
    .groupby("search_term")["ltr_score"]
    .rank(
        method="first",
        ascending=False
    )
)

c:\Users\Kevin\Documents\GitHub\product-search-ranking\.venv\Lib\site-packages\lightgbm\sklearn.py:969: UserWarning: Found 'ndcg_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


In [24]:
test_ndcg3, test_ndcg5 = evaluate_ndcg(test_ltr, "ltr_score")

print(f"LambdaMART NDCG@3: {test_ndcg3:.6f}")
print(f"LambdaMART NDCG@5: {test_ndcg5:.6f}")

LambdaMART NDCG@3: 0.880931
LambdaMART NDCG@5: 0.908926


In [26]:
n3, n5 = evaluate_ndcg(test_ltr, "dense_score")

print(f"Dense NDCG@3: {n3:.6f}")
print(f"Dense NDCG@5: {n5:.6f}")

Dense NDCG@3: 0.881252
Dense NDCG@5: 0.910517


In [28]:
n3, n5 = evaluate_ndcg(test_ltr, "bm25_score")

print(f"BM25 NDCG@3: {n3:.6f}")
print(f"BM25 NDCG@5: {n5:.6f}")

BM25 NDCG@3: 0.882096
BM25 NDCG@5: 0.910985


In [38]:
ltr_score = evaluate_ndcg(test_ltr, "ltr_score")
dense_score = evaluate_ndcg(test_ltr, "dense_score")
bm25_score = evaluate_ndcg(test_ltr, "bm25_score")

scores = pd.DataFrame({
    "BM25": bm25_score,
    "Dense": dense_score,
    "LambdaMART": ltr_score,
}, index=["NDCG@3", "NDCG@5"])
scores

,BM25,Dense,LambdaMART
NDCG@3,0.882096,0.881252,0.880931
NDCG@5,0.910985,0.910517,0.908926


Merge to add product_title and product_text

In [ ]:
train_ltr = pd.merge(
    train_ltr,
    product_catalog[["product_uid", "product_title", "product_text"]],
    on="product_uid",
    how="left"
)


val_ltr = pd.merge(
    val_ltr,
    product_catalog[["product_uid", "product_title", "product_text"]],
    on="product_uid",
    how="left"
)


test_ltr = pd.merge(
    test_ltr,
    product_catalog[["product_uid", "product_title", "product_text"]],
    on="product_uid",
    how="left"
)

Add features:
- token_overlap
- Product of bm25_score and dense_score
- Is all words in the title

In [ ]:
def token_overlap(row):
    query_tokens = set(tokenize(row["search_term"]))
    title_tokens = set(tokenize(row["product_title"]))

    if not query_tokens:
        return 0.0

    return len(query_tokens & title_tokens) / len(query_tokens)

def all_words_in_title(query, title):
    query_tokens = set(tokenize(query))
    title_tokens = set(tokenize(title))

    if not query_tokens:
        return 0

    return int(query_tokens.issubset(title_tokens))

def jaccard_sim(str1, str2):
    a = set(tokenize(str1))
    b = set(tokenize(str2))

    intersection = a & b
    union = a | b

    return len(intersection) / len(union) if union else 0.0

In [ ]:
def add_features(rank_split):
    rank_split["token_overlap"] = rank_split.apply(token_overlap, axis=1)

    rank_split["bm25_dense_product"] = rank_split["bm25_score"] * rank_split["dense_score"]

    rank_split["all_words_in_title"] = [
    all_words_in_title(q, t)
    for q, t in zip(
        rank_split["search_term"],
        rank_split["product_title"]
    )
]

    rank_split["jaccard_sim"] = rank_split.apply(
        lambda row: jaccard_sim(
            row["search_term"],
            row["product_title"]
        ),
        axis=1
    )
    rank_split["length_ratio"] = [
        len(tokenize(q)) / (1+len(tokenize(t)))
        for q, t in zip(rank_split["search_term"], rank_split["product_title"])
    ]

In [129]:
splits = [train_ltr, val_ltr, test_ltr]
for rank_split in splits:
    add_features(rank_split)

In [130]:
features = [
    "bm25_score",
    "dense_score",
    "token_overlap",
    "bm25_dense_product",
    "all_words_in_title",
    "jaccard_similarity",
    "jaccard_similarity_title",
]

X_train = train_ltr[features]
y_train = train_ltr["label"]

X_val = val_ltr[features]
y_val = val_ltr["relevance"].map(relevance_to_label)

X_test = test_ltr[features]
y_test = test_ltr["relevance"].map(relevance_to_label)

In [131]:
param_grid = {
    "n_estimators": [200],
    "learning_rate": [0.05],
    "num_leaves": [31],
    "min_child_samples": [50],
}

In [132]:
ranker = lgb.LGBMRanker(
    objective="lambdarank",
    metric="ndcg",
    ndcg_at=[3, 5],
    n_estimators=200,
    learning_rate=0.01,
    num_leaves=31,
    min_child_samples=10,
    random_state=42,
)

ranker.fit(
    X_train,
    y_train,
    group=group_train,
    eval_set=[(X_val, y_val)],
    eval_group=[group_val],
)

c:\Users\Kevin\Documents\GitHub\product-search-ranking\.venv\Lib\site-packages\lightgbm\sklearn.py:969: UserWarning: Found 'ndcg_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")
c:\Users\Kevin\Documents\GitHub\product-search-ranking\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002881 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 980
[LightGBM] [Info] Number of data points in the train set: 57854, number of used features: 7


,learning_rate,0.01
,n_estimators,200
,objective,'lambdarank'
,min_child_samples,10
,random_state,42
,metric,'ndcg'
,ndcg_at,"[3, 5]"
,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,subsample_for_bin,200000


In [133]:
val_ltr = val_ltr.copy()

val_ltr["ltr_score"] = ranker.predict(
    val_ltr[features]
)

c:\Users\Kevin\Documents\GitHub\product-search-ranking\.venv\Lib\site-packages\lightgbm\sklearn.py:969: UserWarning: Found 'ndcg_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


In [134]:
val_ndcg3, val_ndcg5 = evaluate_ndcg(val_ltr, "ltr_score")

print(f"LambdaMART NDCG@3: {val_ndcg3:.6f}")
print(f"LambdaMART NDCG@5: {val_ndcg5:.6f}")

LambdaMART NDCG@3: 0.894025
LambdaMART NDCG@5: 0.919164


In [135]:
test_ltr["ltr_score"] = ranker.predict(
    test_ltr[features]
)

test_ltr["ltr_rank"] = (
    test_ltr
    .groupby("search_term")["ltr_score"]
    .rank(
        method="first",
        ascending=False
    )
)

c:\Users\Kevin\Documents\GitHub\product-search-ranking\.venv\Lib\site-packages\lightgbm\sklearn.py:969: UserWarning: Found 'ndcg_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


In [136]:
ltr_addon = evaluate_ndcg(test_ltr, "ltr_score")

scores["LambdaMART + Features"] = ltr_addon
scores

,BM25,Dense,LambdaMART,LambdaMART + Features
NDCG@3,0.882096,0.881252,0.880931,0.898981
NDCG@5,0.910985,0.910517,0.908926,0.922563
